# Vehicle Damage Detection — Training + AWS Lambda Deployment
**Model:** EfficientNet-B0 (timm)  
**Output:** `model.tar.gz` -> uploaded to S3 -> served via AWS Lambda Container + API Gateway  
**Pipeline:** Google Colab -> S3 -> ECR -> Lambda -> API Gateway


##  Step 1  Install Dependencies

In [ ]:
!pip install torch torchvision timm scikit-learn matplotlib seaborn --quiet

## Step 2 — Mount Google Drive (for saving outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — Upload & Unzip Dataset

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Upload your Kaggle ZIP here
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/car_damage_data')

print("Files extracted:")
for root, dirs, files_list in os.walk('/content/car_damage_data'):
    for f in files_list[:3]:
        print(os.path.join(root, f))

## Step 4 — Dataset Setup & DataLoaders

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

train_path = '/content/car_damage_data/data1a/training'
val_path   = '/content/car_damage_data/data1a/validation'

print("Training classes:", os.listdir(train_path))
print("Validation classes:", os.listdir(val_path))

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_path, transform=transform_train)
val_dataset   = datasets.ImageFolder(val_path,   transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes   # e.g. ['00-damage', '01-whole']
num_classes = len(class_names)

print(f"Classes     : {class_names}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")

## Step 5 — Load EfficientNet-B0 (pretrained)

In [ ]:
import timm
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Model ready — {num_classes} classes: {class_names}")

## Step 6 — Train the Model

In [ ]:
import time

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total

EPOCHS = 10
best_val_acc = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    start = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)
    scheduler.step()
    elapsed = time.time() - start

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
        print(f"  ✅ New best saved ({val_acc:.1f}%)")

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.1f}% | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.1f}% | "
          f"Time: {elapsed:.1f}s")

print(f"\nBest validation accuracy: {best_val_acc:.1f}%")

## Step 7 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss', color='#E85D24')
ax1.plot(history['val_loss'],   label='Val Loss',   color='#185FA5')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Accuracy', color='#E85D24')
ax2.plot(history['val_acc'],   label='Val Accuracy',   color='#185FA5')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: /content/training_curves.png")

## Step 8 — Evaluate: Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import numpy as np

# Load best weights for evaluation
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Vehicle Damage Detection')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: /content/confusion_matrix.png")

## Step 9 — Write `inference.py` (Lambda handler)
This is the file that goes inside `code/` in the tar.gz.  
It rebuilds the model architecture from scratch and loads the weights — this is what was missing before.


In [ ]:
import os

INFERENCE_CODE = '''
import json, os, io, base64, tarfile, logging
import boto3
import torch
import timm
from torchvision import transforms
from PIL import Image

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Config from Lambda environment variables
BUCKET_NAME  = os.environ["BUCKET_NAME"]
MODEL_KEY    = os.environ.get("MODEL_KEY", "model.tar.gz")
LOCAL_TAR    = "/tmp/model.tar.gz"
EXTRACT_PATH = "/tmp/model"
WEIGHTS_FILE = os.path.join(EXTRACT_PATH, "model.pth")

# Module-level cache: persists across warm Lambda invocations
_model       = None
_class_names = None
_device      = torch.device("cpu")   # Lambda has no GPU

_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Load model (called once on cold start) 
def _load_model():
    global _model, _class_names
    if _model is not None:
        return  # warm invocation — skip

    if not os.path.exists(WEIGHTS_FILE):
        logger.info("Cold start: downloading model from S3...")
        s3 = boto3.client("s3")
        s3.download_file(BUCKET_NAME, MODEL_KEY, LOCAL_TAR)
        os.makedirs(EXTRACT_PATH, exist_ok=True)
        with tarfile.open(LOCAL_TAR) as tar:
            tar.extractall(EXTRACT_PATH)
        logger.info("Extraction complete.")

    checkpoint   = torch.load(WEIGHTS_FILE, map_location=_device)
    _class_names = checkpoint["class_names"]
    num_classes  = checkpoint["num_classes"]

    # Rebuild EfficientNet-B0 architecture, then load trained weights
    _model = timm.create_model("efficientnet_b0",
                                pretrained=False,
                                num_classes=num_classes)
    _model.load_state_dict(checkpoint["model_state_dict"])
    _model.to(_device).eval()
    logger.info(f"Model ready. Classes: {_class_names}")

# Lambda entry point
def lambda_handler(event, context):
    try:
        _load_model()

        # Accept base64-encoded image in JSON body: {"image": "<base64>"}
        body = event.get("body", "")
        if event.get("isBase64Encoded"):
            img_bytes = base64.b64decode(body)
        elif isinstance(body, str):
            data      = json.loads(body)
            img_bytes = base64.b64decode(data["image"])
        else:
            img_bytes = base64.b64decode(body["image"])

        img    = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        tensor = _transform(img).unsqueeze(0).to(_device)

        with torch.no_grad():
            logits = _model(tensor)
            probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

        predicted_class = _class_names[int(probs.argmax())]
        confidence      = float(probs.max())

        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({
                "predicted_class": predicted_class,
                "confidence":      round(confidence, 4),
                "scores": {
                    name: round(float(p), 4)
                    for name, p in zip(_class_names, probs)
                }
            })
        }

    except Exception as e:
        logger.exception("Inference failed")
        return {
            "statusCode": 500,
            "body": json.dumps({"error": str(e)})
        }
'''

os.makedirs('/content/model_package/code', exist_ok=True)
with open('/content/model_package/code/inference.py', 'w') as f:
    f.write(INFERENCE_CODE.strip())

print(" inference.py written to /content/model_package/code/inference.py")


## Step 10 — Package `model.tar.gz`
This creates the final tar.gz with the correct structure:


In [ ]:
import tarfile, shutil

# Load best weights back into model
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()

# Save complete checkpoint: weights + metadata
model_info = {
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'num_classes':      num_classes,
}
torch.save(model_info, '/content/model_package/model.pth')
print("model.pth saved (weights + class_names + num_classes)")

# Package everything into tar.gz
with tarfile.open('/content/model.tar.gz', 'w:gz') as tar:
    tar.add('/content/model_package', arcname='.')

# Verify contents
print("\n tar.gz contents:")
with tarfile.open('/content/model.tar.gz', 'r:gz') as tar:
    for m in sorted(tar.getmembers(), key=lambda x: x.name):
        size_str = f"{m.size:>10,} bytes" if m.isfile() else "         <dir>"
        print(f"  {m.name:50s}  {size_str}")

print("\n model.tar.gz is ready")

## Step 11 — Save Everything to Google Drive & Download

In [ ]:
import shutil

drive_dir = '/content/drive/MyDrive/vehicle_damage_deploy'
os.makedirs(drive_dir, exist_ok=True)

# Copy all outputs
files_to_save = {
    '/content/model.tar.gz':          f'{drive_dir}/model.tar.gz',
    '/content/training_curves.png':   f'{drive_dir}/training_curves.png',
    '/content/confusion_matrix.png':  f'{drive_dir}/confusion_matrix.png',
}
for src, dst in files_to_save.items():
    shutil.copy(src, dst)
    print(f" Saved: {dst}")

print("\nDownloading to your computer...")
from google.colab import files
files.download('/content/model.tar.gz')
files.download('/content/training_curves.png')
files.download('/content/confusion_matrix.png')

print("\n" + "="*55)
print("  ALL DONE — next steps on your local machine:")
print("="*55)
print("  1. Download 'vehicle_damage_deploy' folder from Drive")
print("     It contains: Dockerfile, inference.py,")
print("                  requirements.txt, deploy.sh, model.tar.gz")
print("  2. Put model.tar.gz in the same folder")
print("  3. Run:  chmod +x deploy.sh && ./deploy.sh")
print("  4. The script handles S3, ECR, Lambda, API Gateway")
print("="*55)